## 1. Data Processing

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import shap
import gpboost as gpb

In [ ]:
'''Load the data'''
icu_data = pd.read_csv('./df.csv')
icu_data.set_index('PtID', inplace = True)
icu_data.head()

In [ ]:
print(icu_data.shape)
icu_data.info()

In [ ]:
'''
data cleaning:
    handle text and categorial attributes
'''
# I think one-hot encoding is probably better, but it's hard to interpret the variable importance, etc.
# Label Encoder might lead to the algorithm placing more importance on larger number, say 5 for {1, 2, 3, 4, 5}
# I think facotr() in r is similiar to one-hot encoding

idx = icu_data.select_dtypes(include = "object").columns
idx = idx.delete(0)
le = LabelEncoder()
icu_data[idx] = icu_data[idx].apply(le.fit_transform)

icu_data.head()

In [ ]:
icu_data.info()

In [ ]:
'''Responses and covariates'''

# response: delta sofa score
delta_sofa = icu_data[['DeltaSOFA']]

# covariates
idx = icu_data.columns
idx = idx.delete([idx.get_loc('DocID'), idx.get_loc('DischargeStatus'), idx.get_loc('LengthOfStay'),
                  idx.get_loc('DeltaSOFA'), idx.get_loc('SOFA')])
covariates = icu_data[idx]

## 2. Random Forest

In [ ]:
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 1000, num = 5)]
# Number of features to consider at every split, ESL suggests p/3 for regression RF
max_features = [7, 9, 11]
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 100, num = 10)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf}
random_grid

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestRegressor()
# Random search of parameters, using 5 fold cross validation,
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = random_grid, n_iter = 100, cv = 5, verbose=1, random_state=123, n_jobs = -1)
# Fit the random search model
rf_random.fit(covariates, delta_sofa.values.ravel())
rf_random.best_params_

In [ ]:
#Random search allowed us to narrow down the range for each hyperparameter. Now that we know where to concentrate our search, we can explicitly specify every combination of settings to try.
# Create the parameter grid based on the results of random search
param_grid = {
    'n_estimators': [300, 400, 500],
    'max_features': [7, 9, 11],
    'max_depth': [70, 80, 90],
    'min_samples_leaf': [3, 4, 5],
    'min_samples_split': [5, 10]
}
# Instantiate the grid search model
grid_search = GridSearchCV(estimator = rf, param_grid = param_grid, cv = 5, n_jobs = -1, verbose = 1)
grid_search.fit(covariates, delta_sofa.values.ravel())
grid_search.best_params_

In [ ]:
# train the model
rf = RandomForestRegressor(n_estimators=300, max_features=9, max_depth=70, min_samples_leaf=5,
                              min_samples_split=5, random_state=123)
rf.fit(covariates, delta_sofa.values.ravel())

# get shap values
explainer = shap.explainers.Tree(rf, covariates)
shap_values = explainer(covariates)

In [ ]:
shap.summary_plot(
    shap_values, covariates, plot_type="bar", show=False
)
plt.xlabel("mean(|SHAP value|)")
plt.show()

In [ ]:
shap.summary_plot(shap_values, covariates, show=False)
plt.show()

In [ ]:
rf_shap_interaction = shap.TreeExplainer(rf).shap_interaction_values(covariates)
shap.dependence_plot(("APACHEII", 'PatientAge'), rf_shap_interaction, covariates, show=False)
plt.show()

In [ ]:
#https://github.com/suinleelab/treeexplainer-study/blob/master/notebooks/mortality/NHANES%20I%20Analysis.ipynb

## 3. Xgboost

In [ ]:
# Hyperparameters tuning
# https://xgboost.readthedocs.io/en/latest/tutorials/param_tuning.html
# https://www.analyticsvidhya.com/blog/2016/03/complete-guide-parameter-tuning-xgboost-with-codes-python/
random_grid = {
 'max_depth': [1, 2, 4, 6, 8, 10],
 'min_child_weight':range(1,6,2),
 'gamma': [i/10.0 for i in range(0,6)],
 'subsample': [i/10.0 for i in range(5,10)],
 'colsample_bytree': [i/10.0 for i in range(5,10)],
 # control the model complexity
 'reg_alpha':[1e-5, 1e-3, 1e-2, 0.1, 1, 10]
}
warnings.filterwarnings("ignore")
# Random search of parameters, using 5 fold cross validation,
# search across 100 different combinations, and use all available cores
xgb_random = RandomizedSearchCV(estimator = xgb.XGBRegressor(seed=123), param_distributions = random_grid, n_iter = 50,
                                cv = 5, verbose=0, random_state=123, n_jobs = -1)
# Fit the random search model
xgb_random.fit(covariates, delta_sofa.values.ravel())
xgb_random.best_params_

In [ ]:
# Create the parameter grid based on the results of random search
param_grid = {
     'max_depth':[2, 3],
     'min_child_weight':[1, 2, 3],
     'gamma': [0.2, 0.4, 0.6],
     'subsample': [0.8, 0.9],
     'colsample_bytree': [0.8, 0.9],
     # control the model complexity
     'reg_alpha':[0.001, 0.01, 0.1]
}
# Instantiate the grid search model
warnings.filterwarnings("ignore")
grid_search = GridSearchCV(estimator = xgb.XGBRegressor(seed=123), param_grid = param_grid,
                          cv = 5, n_jobs = -1, verbose = 0)
grid_search.fit(covariates, delta_sofa.values.ravel())
grid_search.best_params_

In [ ]:
'''train the model'''
xgb_model = xgb.XGBRegressor(colsample_bytree=0.9, gamma=0.2, max_depth=2, min_child_weight=3,
 reg_alpha=0.01, subsample=0.9, random_state=123, verbosity=0)
xgb_model.fit(covariates, delta_sofa.values.ravel())

In [ ]:
'''SHAP'''

# define the explainer object
explainer = shap.TreeExplainer(xgb_model)


# calculate the shap values: resampling the training dataset and calculating the impact over these perturbations
xgb_shap = explainer.shap_values(covariates)

In [ ]:
shap.summary_plot(
    xgb_shap, covariates, plot_type="bar", show=False
)
plt.xlabel("mean(|SHAP value|)")
plt.show()

In [ ]:
shap.summary_plot(xgb_shap, covariates, show=False)
plt.show()

## 4. Mixed Effect Boosting Model

In [ ]:
#https://towardsdatascience.com/tree-boosted-mixed-effects-models-4df610b624cb
# group by ICU

In [ ]:
idx = idx.delete([idx.get_loc('DocID'), idx.get_loc('DischargeStatus'), idx.get_loc('LengthOfStay'),
                  idx.get_loc('DeltaSOFA'), idx.get_loc('SOFA'), idx.get_loc('Department')])
covariates = icu_data[idx]

In [ ]:
gp_model = gpb.GPModel(group_data=covariates[['Department']])
# create dataset for gpb.train function
data = gpb.Dataset(covariates, delta_sofa)

# larger grid and random search
params = {'objective': 'regression_l2', 'verbose': 0}

param_grid_large = {'learning_rate': [1, 0.5, 0.1, 0.05, 0.01],
                    'min_data_in_leaf': [5, 10, 20, 50, 100, 200],
                    'max_depth': [1, 2, 4, 6, 8, 10]}
opt_params = gpb.grid_search_tune_parameters(param_grid=param_grid_large,
                                             params=params,
                                             num_try_random=20,
                                             nfold=5,
                                             gp_model=gp_model,
                                             use_gp_model_for_validation=True,
                                             train_set=data,
                                             verbose_eval=0,
                                             num_boost_round=1000,
                                             early_stopping_rounds=10,
                                             seed=123)
print("Best number of iterations: " + str(opt_params['best_iter']))
print("Best score: " + str(opt_params['best_score']))
print("Best parameters: " + str(opt_params['best_params']))

In [ ]:
# Small grid and deterministic search
# Change the numbers using the result from random search
param_grid_small = {'learning_rate': [1, .5, 0.1, 0.01], 'min_data_in_leaf': [75, 100, 125],
                    'max_depth': [6, 8, 10]}
opt_params = gpb.grid_search_tune_parameters(param_grid=param_grid_small,
                                             params=params,
                                             num_try_random=None,
                                             nfold=5,
                                             gp_model=gp_model,
                                             use_gp_model_for_validation=True,
                                             train_set=data,
                                             verbose_eval=0,
                                             num_boost_round=1000,
                                             early_stopping_rounds=10,
                                             seed=123)
print("Best number of iterations: " + str(opt_params['best_iter']))
print("Best score: " + str(opt_params['best_score']))
print("Best parameters: " + str(opt_params['best_params']))

In [ ]:
# Do not include random effect predictions for validation (observe the higher test error)
cvbst = gpb.cv(params=params, train_set=data,
               gp_model=gp_model, use_gp_model_for_validation=False,
               num_boost_round=100, early_stopping_rounds=5,
               nfold=2, verbose_eval=True, show_stdv=False, seed=1)
print("Best number of iterations: " + str(np.argmin(cvbst['l2-mean'])))

In [ ]:
# Define and train GPModel
# specify tree-boosting parameters as a dict
params = {'objective': 'regression_l2', 'learning_rate': 0.5, 'max_depth': 8, 'min_data_in_leaf': 125,
          'verbose': 0}
# do cross-validation to find best number of iterations
cvbst = gpb.cv(params=params, train_set=data,
               gp_model=gp_model, use_gp_model_for_validation=True,
               num_boost_round=100, early_stopping_rounds=5,
               nfold=5, verbose_eval=True, show_stdv=False, seed=123)
print("Best number of iterations: " + str(np.argmin(cvbst['l2-mean'])))

In [ ]:
gpbmodel = gpb.train(params=params, train_set=data, gp_model=gp_model, num_boost_round=26)
gp_model.summary() # estimated covariance parameters

In [ ]:
# define the explainer object
explainer = shap.TreeExplainer(gpbmodel)

# calculate the shap values: resampling the training dataset and calculating the impact over these perturbations
bst_shap = explainer.shap_values(covariates)

In [ ]:
shap.summary_plot(bst_shap, covariates, plot_type="bar", show=False
)
plt.xlabel("mean(|SHAP value|)")
plt.show()

In [ ]:
shap.summary_plot(bst_shap, covariates, show=False)
plt.show()

In [ ]:
#https://slundberg.github.io/shap/notebooks/plots/dependence_plot.html
shap.dependence_plot("APACHEII", bst_shap, covariates, interaction_index="PhysicianAveSc")

In [ ]:
# decoding relationship
X_cat = covariates.copy()
relationship_decoding = {
    0: "greater than 60",
    1: "less than 60"
}
.....

## 6. Mxied Random Forest

In [ ]:
# could ignore, revise the code first....
from merf import MERF
rf_params={'max_depth': 6, 'n_estimators': 300}
merf_model = MERF(max_iterations=100, rf_params=rf_params)
print("Warning: the following takes a lot of time")
start_time = time.time() # measure time
merf_model.fit(pd.DataFrame(X_train), np.ones(shape=(ntrain,1)), pd.Series(group_train), y_train)
results.loc["MERF","Time"] = time.time() - start_time
y_pred = merf_model.predict(pd.DataFrame(X_test), np.ones(shape=(ntrain,1)), pd.Series(group_test))
results.loc["MERF","RMSE"] = np.sqrt(np.mean((y_test - y_pred) ** 2))

print(results.apply(pd.to_numeric).round(3))